# Imports

In [33]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

from autogluon.tabular import TabularDataset, TabularPredictor
from autogluon.multimodal import MultiModalPredictor
import xgboost as xgb

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)

SCORING = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}
GRID_SEARCH_KWARGS = dict(
    cv=10,
    scoring=SCORING,
    refit='r2',
    n_jobs=-1,
    return_train_score=True
)
LINEAR_PARAM_GRID = {
    'model__fit_intercept': [True, False],
    'model__positive': [False, True]
}
TREE_PARAM_GRID = {
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
RF_PARAM_GRID = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
MLP_PARAM_GRID = {
    'model__hidden_layer_sizes': [(100,), (100, 50), (50, 25)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01]
}
XGB_PARAM_GRID = {
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__n_estimators': [50, 100, 200]
}


# Load data

In [34]:
data = pd.read_csv("../data/data_yolo_50e.csv", index_col=0)
print(data)

      country time_of_day        lat       long       road_type  \
8459       SE         day  55.604209  13.028574            city   
63960      DE         day  50.936889   6.908079  arterial-urban   
71801      IT         day  41.987014  12.496538         highway   
85589      IT       night  41.794853  12.522880         highway   
90649      DE         day  49.182153   9.414737         highway   
...       ...         ...        ...        ...             ...   
74962      PL         day  54.378439  18.605984  arterial-urban   
90673      PL         day  50.253217  19.823732   smaller-rural   
24373      DE         day  50.931399   6.953686            city   
41597      SE         day  55.608149  13.003458            city   
40353      IT       night  41.918667  12.383545            city   

      road_condition            weather  solar_angle_elevation  month  hour  \
8459          normal             cloudy              36.560248      5    14   
63960         normal               ra

In [35]:
data = data.dropna().reset_index(drop=True)
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 10,006 rows and 38 columns


In [36]:
iou = data["iou"]
lrp = data["lrp"]
conf = data["conf"]
image_paths_dict = {int(img_pth.split("_")[0]):f"../data/zod_yolo/images/val/{img_pth}" for img_pth in os.listdir("../data/zod_yolo/images/val") if img_pth.endswith(".jpg")}
img_path = pd.DataFrame.from_dict(image_paths_dict, orient='index', columns=["Images"])

data = data.drop(columns=["iou", "lrp"])
data_indices = data.index.to_numpy()

In [37]:
all_columns = data.columns    
all_columns = all_columns.drop(["weather_code", "cloud_cover_low", "cloud_cover_mid"])

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 10006
Columns: Index(['brightness', 'camera_distance_from_ground', 'camera_offset',
       'camera_pitch_angle', 'cloud_cover', 'complexity', 'conf', 'contrast',
       'country', 'distortion_magnitude', 'field_view_horizontal',
       'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat',
       'lateral_acceleration', 'lateral_velocity', 'long', 'month',
       'noisiness', 'quality', 'rain', 'relative_humidity_2m',
       'road_condition', 'road_type', 'sharpness', 'snowfall',
       'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather',
       'wind_speed_10m'],
      dtype='object')
Number of columns: 33


In [38]:
numeric_columns = ['conf', 'solar_angle_elevation', 'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m', 'forward_acceleration', 'lateral_acceleration', 'forward_velocity', 'lateral_velocity', 'field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset', 'laplacian', 'brightness', 'contrast', 'sharpness', 'noisiness', 'quality', 'complexity']

categorical_columns = ["country", "time_of_day", "road_type", "road_condition", "weather"]
other = ["hour", "month", "lat", "long"]
print(sorted(numeric_columns + categorical_columns + other))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns) + len(other)), "Columns not match"

['brightness', 'camera_distance_from_ground', 'camera_offset', 'camera_pitch_angle', 'cloud_cover', 'complexity', 'conf', 'contrast', 'country', 'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration', 'lateral_velocity', 'long', 'month', 'noisiness', 'quality', 'rain', 'relative_humidity_2m', 'road_condition', 'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather', 'wind_speed_10m']


# Assessor Tests

Split data into train 60% and validation 40%

In [39]:
#X_train, X_test, y_train, y_test = train_test_split(data, test_size=0.4, train_size=0.6, stratify=data["iou"])
train_idx, test_idx = train_test_split(data_indices, test_size=0.4, train_size=0.6)

X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]
y_train_lrp = lrp.loc[train_idx]
conf_train = conf.loc[train_idx]
conf_train = pd.DataFrame(conf_train)

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]
y_test_lrp = lrp.loc[test_idx]
conf_test = conf.loc[test_idx]
conf_test = pd.DataFrame(conf_test)

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print(X_train.columns)

X: 6003 4003
y: 6003 4003
Index(['country', 'time_of_day', 'lat', 'long', 'road_type', 'road_condition',
       'weather', 'solar_angle_elevation', 'month', 'hour',
       'forward_acceleration', 'lateral_acceleration', 'forward_velocity',
       'lateral_velocity', 'field_view_horizontal',
       'camera_distance_from_ground', 'camera_pitch_angle',
       'distortion_magnitude', 'camera_offset', 'laplacian', 'quality',
       'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity',
       'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall',
       'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'wind_speed_10m',
       'weather_code', 'conf'],
      dtype='object')


In [40]:
model_results = []

### IoU

#### Baseline

Random Prediction. Mean of metric over training set.

In [41]:
mean_iou_train = np.mean(y_train)
iou_pred_test = np.full_like(y_test, mean_iou_train)
mean_baseline_r2 = r2_score(y_test, iou_pred_test)
mean_baseline_mae = np.mean(np.abs(y_test - iou_pred_test))
mean_baseline_mse = np.mean((y_test - iou_pred_test)**2)
print(f"R2 score of mean baseline: {mean_baseline_r2:.4f}")
print(f"MAE of mean baseline: {mean_baseline_mae:.4f}")
print(f"MSE of mean baseline: {mean_baseline_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Constant Mean Predictor',
    'test_r2': mean_baseline_r2,
    'test_mae': mean_baseline_mae,
    'test_mse': mean_baseline_mse,
})


R2 score of mean baseline: -0.0004
MAE of mean baseline: 0.1227
MSE of mean baseline: 0.0273


#### Linear Reg on Conf

In [42]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_grid = GridSearchCV(
    linear_reg_conf_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_grid.fit(conf_train, y_train)

best_idx = linear_reg_conf_grid.best_index_
mean_train_r2 = linear_reg_conf_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_grid.best_params_}")
best_linear_reg_conf = linear_reg_conf_grid.best_estimator_


/home/felix/.pyenv/versions/3.11.14/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/felix/.pyenv/versions/3.11.14/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/felix/.pyenv/versions/3.11.14/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from 

Mean CV train r2 score 0.3408
Mean CV test r2 score 0.3394
Mean CV train MAE score 0.1014
Mean CV test MAE score 0.1015
Mean CV train MSE score 0.0181
Mean CV test MSE score 0.0181
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [43]:
y_pred = best_linear_reg_conf.predict(conf_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Univariate Linear Regression (Confidence)',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3493
MAE test score 0.0996
MSE test score 0.0177


#### Linear Regression

Train models with cv and then test.

In [44]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_grid = GridSearchCV(
    linear_reg_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_grid.fit(X_train, y_train)

best_idx = linear_reg_grid.best_index_
mean_train_r2 = linear_reg_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_grid.best_params_}")
best_linear_reg = linear_reg_grid.best_estimator_


Mean CV train r2 score 0.3753
Mean CV test r2 score 0.3556
Mean CV train MAE score 0.0992
Mean CV test MAE score 0.1005
Mean CV train MSE score 0.0172
Mean CV test MSE score 0.0177
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [45]:
y_pred = best_linear_reg.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Linear Regression',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3679
MAE test score 0.0984
MSE test score 0.0172


#### Decision Trees

In [46]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_grid = GridSearchCV(
    decision_tree_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_grid.fit(X_train, y_train)

best_idx = decision_tree_grid.best_index_
mean_train_r2 = decision_tree_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_grid.best_params_}")
best_decision_tree = decision_tree_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *ar

Mean CV train r2 score 0.4001
Mean CV test r2 score 0.3323
Mean CV train MAE score 0.0962
Mean CV test MAE score 0.1015
Mean CV train MSE score 0.0165
Mean CV test MSE score 0.0184
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


In [47]:
y_pred = best_decision_tree.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Decision Tree',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3425
MAE test score 0.0997
MSE test score 0.0179


#### Random Forest

In [48]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_grid = GridSearchCV(
    rand_forest_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_grid.fit(X_train, y_train)

best_idx = rand_forest_grid.best_index_
mean_train_r2 = rand_forest_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_grid.best_params_}")
best_rand_forest = rand_forest_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *ar

Mean CV train r2 score 0.4843
Mean CV test r2 score 0.3627
Mean CV train MAE score 0.0900
Mean CV test MAE score 0.0990
Mean CV train MSE score 0.0142
Mean CV test MSE score 0.0175
Best params: {'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [49]:
y_pred = best_rand_forest.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Random Forest',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3772
MAE test score 0.0969
MSE test score 0.0170


#### MLP

In [50]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_grid = GridSearchCV(
    mlp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_grid.fit(X_train, y_train)

best_idx = mlp_grid.best_index_
mean_train_r2 = mlp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_grid.best_params_}")
best_mlp = mlp_grid.best_estimator_


Mean CV train r2 score 0.3790
Mean CV test r2 score 0.2738
Mean CV train MAE score 0.0999
Mean CV test MAE score 0.1069
Mean CV train MSE score 0.0171
Mean CV test MSE score 0.0199
Best params: {'model__activation': 'tanh', 'model__alpha': 0.01, 'model__hidden_layer_sizes': (50, 25), 'model__learning_rate_init': 0.001}


In [51]:
y_pred = best_mlp.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'MLP',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3279
MAE test score 0.1023
MSE test score 0.0183


#### XGBoost

In [52]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_grid.fit(X_train, y_train)


best_idx = xgb_grid.best_index_
mean_train_r2 = xgb_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_grid.best_params_}")
best_xgb = xgb_grid.best_estimator_

Mean CV train r2 score 0.5142
Mean CV test r2 score 0.3464
Mean CV train MAE score 0.0898
Mean CV test MAE score 0.1014
Mean CV train MSE score 0.0134
Mean CV test MSE score 0.0179
Best params: {'model__learning_rate': 0.01, 'model__n_estimators': 200}


In [53]:
y_pred = best_xgb.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'XGBoost',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3618
MAE test score 0.0997
MSE test score 0.0174


#### Autogluon

##### Tabluar

In [54]:
train = pd.merge(X_train, y_train, left_index=True, right_index=True)

In [55]:
autoglue_model = TabularPredictor(label="iou", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/iou/")
autoglue_predictor = autoglue_model.fit(train)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.1b20251210
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #88~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Oct 14 14:03:14 UTC 2
CPU Count:          24
Pytorch Version:    2.7.1+cu126
CUDA Version:       12.6
GPU Memory:         GPU 0: 23.69/23.69 GB | GPU 1: 23.69/23.69 GB
Total GPU Memory:   Free: 47.38 GB, Allocated: 0.00 GB, Total: 47.38 GB
GPU Count:          2
Memory Avail:       93.57 GB / 125.48 GB (74.6%)
Disk Space Avail:   9.36 GB / 3665.96 GB (0.3%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme

In [56]:
autoglue_predictor.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.377960          r2       0.026218  9.081975                0.000578           0.061529            2       True         10
1             CatBoost   0.371956          r2       0.005345  2.193675                0.005345           2.193675            1       True          4
2              XGBoost   0.371115          r2       0.005697  0.669673                0.005697           0.669673            1       True          7
3             LightGBM   0.361793          r2       0.002449  0.752486                0.002449           0.752486            1       True          2
4        LightGBMLarge   0.355660          r2       0.003238  1.992536                0.003238           1.992536            1       True          9
5           LightGBMXT   0.354881          r

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'NeuralNetFastAI': 'NNFastAiTabularModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.3548807034755592,
  'LightGBM': 0.3617932866542023,
  'RandomForestMSE': 0.3545027243775368,
  'CatBoost': 0.37195591135028083,
  'ExtraTreesMSE': 0.3530946236191813,
  'NeuralNetFastAI': 0.3293915388757144,
  'XGBoost': 0.3711150623557782,
  'NeuralNetTorch': 0.35148245701466785,
  'LightGBMLarge': 0.3556595511810794,
  'WeightedEnsemble_L2': 0.3779598952705122},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  'Ne

In [ ]:
y_pred = autoglue_predictor.predict(X_test)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Autogluon_Tab',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


R2 test score 0.3863
MAE test score 0.0970
MSE test score 0.0167


##### Images

In [58]:
X_train_img = pd.merge(X_train, img_path, left_index=True, right_index=True)
X_test_img = pd.merge(X_test, img_path, how="left", right_index=True, left_index=True)
train_img = pd.merge(X_train_img, y_train, left_index=True, right_index=True)

In [60]:
autoglue_model_img = MultiModalPredictor(label="iou", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/iou/img/")
hyperparams = {
    "env.strategy": "ddp_notebook",
    "env.num_gpus": 1,
    "env.num_workers": 0,
    #"optim.max_epochs": 15,
    #"optim.patience": 3,
    #"optim.learning_rate": 1e-4,
    #"optim.weight_decay": 1e-4,
    #"optim.warmup_steps": 500,
    #"env.per_gpu_batch_size": 8,
    #"env.batch_size": 128,
    #"model.timm_image.checkpoint_name": "convnext_base",
}
autoglue_predictor_img = autoglue_model_img.fit(train_img, hyperparameters=hyperparams)

=================== System Info ===================
AutoGluon Version:  1.4.1b20251210
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #88~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Oct 14 14:03:14 UTC 2
CPU Count:          24
Pytorch Version:    2.7.1+cu126
CUDA Version:       12.6
GPU Memory:         GPU 0: 23.69/23.69 GB | GPU 1: 23.69/23.69 GB
Total GPU Memory:   Free: 47.38 GB, Allocated: 0.00 GB, Total: 47.38 GB
GPU Count:          2
Memory Avail:       92.76 GB / 125.48 GB (73.9%)
Disk Space Avail:   10.27 GB / 3665.96 GB (0.3%)

AutoMM starts to create your model. ✨✨✨

To track the learning progress, you can open a terminal and launch Tensorboard:
    ```shell
    # Assume you have installed tensorboard
    tensorboard --logdir /home/felix/Predictability-AD/models/assessors/autoglue_tab/iou/img
    ```

Seed set to 0
GPU Count: 2
GPU Count to be Used: 1

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Epoch 0:  50%|█████     | 29/58 [00:07<00:07,  3.87it/s]

...

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Predicting DataLoader 0: 100%|██████████| 4/4 [00:00<00:00,  7.87it/s]


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Predicting DataLoader 0: 100%|██████████| 4/4 [00:00<00:00,  8.10it/s]


AutoMM has created your model. 🎉🎉🎉

To load the model, use the code below:
    ```python
    from autogluon.multimodal import MultiModalPredictor
    predictor = MultiModalPredictor.load("/home/felix/Predictability-AD/models/assessors/autoglue_tab/iou/img")
    ```

If you are not satisfied with the model, try to increase the training time, 
adjust the hyperparameters (https://auto.gluon.ai/stable/tutorials/multimodal/advanced_topics/customization.html),
or post issues on GitHub (https://github.com/autogluon/autogluon/issues).




In [61]:
autoglue_predictor_img.fit_summary(verbosity=2)

{'val_r2': -0.25644969940185547, 'training_time': 170.8363802433014}

In [ ]:
y_pred = autoglue_predictor_img.predict(X_test_img)
iou_test_r2 = r2_score(y_test, y_pred)
iou_test_mae = np.mean(np.abs(y_test - y_pred))
iou_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {iou_test_r2:.4f}")
print(f"MAE test score {iou_test_mae:.4f}")
print(f"MSE test score {iou_test_mse:.4f}")
model_results.append({
    'target': 'IoU',
    'model': 'Autogluon_Mult',
    'test_r2': iou_test_r2,
    'test_mae': iou_test_mae,
    'test_mse': iou_test_mse,
})


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Predicting DataLoader 0: 100%|██████████| 126/126 [00:12<00:00,  9.74it/s]
5425    0.373649
287     0.436636
2813    0.390407
6835    0.374257
8545    0.420907
          ...   
9568    0.463245
221     0.394709
2646    0.353684
8257    0.413019
615     0.413349
Name: iou, Length: 4003, dtype: float32 5425    0.365499
287     0.331839
2813    0.485006
6835    0.324405
8545    0.409719
          ...   
9568    0.352842
221     0.000000
2646    0.512610
8257    0.316860
615     0.682589
Name: iou, Length: 4003, dtype: float64
R2 test score 0.0210
MAE test score 0.1223
MSE test score 0.0267


### LRP


#### Baseline

Predict Only the Mean


In [63]:
mean_lrp_train = np.mean(y_train_lrp)
lrp_pred_test = np.full_like(y_test_lrp, mean_lrp_train)
mean_lrp_r2 = r2_score(y_test_lrp, lrp_pred_test)
mean_lrp_mae = np.mean(np.abs(y_test_lrp - lrp_pred_test))
mean_lrp_mse = np.mean((y_test_lrp - lrp_pred_test)**2)
print(f"R2 score of random baseline: {mean_lrp_r2:.4f}")
print(f"MAE of random baseline: {mean_lrp_mae:.4f}")
print(f"MSE of random baseline: {mean_lrp_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Constant Mean Predictor',
    'test_r2': mean_lrp_r2,
    'test_mae': mean_lrp_mae,
    'test_mse': mean_lrp_mse,
})


R2 score of random baseline: -0.0002
MAE of random baseline: 0.1054
MSE of random baseline: 0.0206


#### Linear Reg on Conf

In [64]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_lrp_grid = GridSearchCV(
    linear_reg_conf_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_lrp_grid.fit(conf_train, y_train_lrp)

best_idx = linear_reg_conf_lrp_grid.best_index_
mean_train_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_lrp_grid.best_params_}")
best_linear_reg_conf_lrp = linear_reg_conf_lrp_grid.best_estimator_


/home/felix/.pyenv/versions/3.11.14/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/felix/.pyenv/versions/3.11.14/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/felix/.pyenv/versions/3.11.14/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from 

Mean CV train r2 score 0.3990
Mean CV test r2 score 0.3975
Mean CV train MAE score 0.0827
Mean CV test MAE score 0.0827
Mean CV train MSE score 0.0126
Mean CV test MSE score 0.0127
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [65]:
y_pred = best_linear_reg_conf_lrp.predict(conf_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Univariate Linear Regression (Confidence)',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.3946
MAE test score 0.0822
MSE test score 0.0125


#### Linear Regression


Train models with cv and then test.


In [66]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_lrp_grid = GridSearchCV(
    linear_reg_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_lrp_grid.fit(X_train, y_train_lrp)

best_idx = linear_reg_lrp_grid.best_index_
mean_train_r2 = linear_reg_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_lrp_grid.best_params_}")
best_linear_reg_lrp = linear_reg_lrp_grid.best_estimator_


Mean CV train r2 score 0.4405
Mean CV test r2 score 0.4199
Mean CV train MAE score 0.0796
Mean CV test MAE score 0.0808
Mean CV train MSE score 0.0118
Mean CV test MSE score 0.0122
Best params: {'model__fit_intercept': False, 'model__positive': False}


In [67]:
y_pred = best_linear_reg_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Linear Regression',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4208
MAE test score 0.0802
MSE test score 0.0119


#### Decision Trees


In [68]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_lrp_grid = GridSearchCV(
    decision_tree_lrp_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_lrp_grid.fit(X_train, y_train_lrp)

best_idx = decision_tree_lrp_grid.best_index_
mean_train_r2 = decision_tree_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_lrp_grid.best_params_}")
best_decision_tree_lrp = decision_tree_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *ar

Mean CV train r2 score 0.4632
Mean CV test r2 score 0.4041
Mean CV train MAE score 0.0764
Mean CV test MAE score 0.0802
Mean CV train MSE score 0.0113
Mean CV test MSE score 0.0125
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


In [69]:
y_pred = best_decision_tree_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Decision Tree',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4108
MAE test score 0.0785
MSE test score 0.0121


#### Random Forest


In [70]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_lrp_grid = GridSearchCV(
    rand_forest_lrp_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_lrp_grid.fit(X_train, y_train_lrp)

best_idx = rand_forest_lrp_grid.best_index_
mean_train_r2 = rand_forest_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_lrp_grid.best_params_}")
best_rand_forest_lrp = rand_forest_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/base.py", line 1363, in wrapper
    return fit_method(estimator, *ar

Mean CV train r2 score 0.5521
Mean CV test r2 score 0.4354
Mean CV train MAE score 0.0704
Mean CV test MAE score 0.0780
Mean CV train MSE score 0.0094
Mean CV test MSE score 0.0118
Best params: {'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [71]:
y_pred = best_rand_forest_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Random Forest',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4329
MAE test score 0.0772
MSE test score 0.0117


#### MLP


In [72]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_lrp_grid = GridSearchCV(
    mlp_lrp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_lrp_grid.fit(X_train, y_train_lrp)

best_idx = mlp_lrp_grid.best_index_
mean_train_r2 = mlp_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_lrp_grid.best_params_}")
best_mlp_lrp = mlp_lrp_grid.best_estimator_


Mean CV train r2 score 0.3630
Mean CV test r2 score 0.3370
Mean CV train MAE score 0.0857
Mean CV test MAE score 0.0870
Mean CV train MSE score 0.0134
Mean CV test MSE score 0.0139
Best params: {'model__activation': 'relu', 'model__alpha': 0.001, 'model__hidden_layer_sizes': (50, 25), 'model__learning_rate_init': 0.01}


In [73]:
y_pred = best_mlp_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'MLP',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4052
MAE test score 0.0805
MSE test score 0.0123


#### XGBoost

In [74]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_lrp_grid = GridSearchCV(
    xgb_lrp_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_lrp_grid.fit(X_train, y_train_lrp)


best_idx = xgb_lrp_grid.best_index_
mean_train_r2 = xgb_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_lrp_grid.best_params_}")
best_xgb_lrp = xgb_lrp_grid.best_estimator_

Mean CV train r2 score 0.6028
Mean CV test r2 score 0.4192
Mean CV train MAE score 0.0693
Mean CV test MAE score 0.0797
Mean CV train MSE score 0.0084
Mean CV test MSE score 0.0122
Best params: {'model__learning_rate': 0.01, 'model__n_estimators': 200}


In [75]:
y_pred = best_xgb_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test_lrp - y_pred))
lrp_test_mse = np.mean((y_test_lrp - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'XGBoost',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4130
MAE test score 0.0794
MSE test score 0.0121


#### Autogluon

##### Tabluar

In [76]:
train_lrp = pd.merge(X_train, y_train_lrp, left_index=True, right_index=True)

In [77]:
autoglue_model_lrp = TabularPredictor(label="lrp", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/lrp/tab/")
autoglue_predictor_lrp = autoglue_model_lrp.fit(train_lrp)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.1b20251210
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #88~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Oct 14 14:03:14 UTC 2
CPU Count:          24
Pytorch Version:    2.7.1+cu126
CUDA Version:       12.6
GPU Memory:         GPU 0: 22.61/23.69 GB | GPU 1: 23.69/23.69 GB
Total GPU Memory:   Free: 46.30 GB, Allocated: 1.08 GB, Total: 47.38 GB
GPU Count:          2
Memory Avail:       92.59 GB / 125.48 GB (73.8%)
Disk Space Avail:   9.91 GB / 3665.96 GB (0.3%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme

	Fitting with cpus=12, gpus=0, mem=0.0/92.6 GB
	0.438	 = Validation score   (r2)
	1.1s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: LightGBM ...
	Fitting with cpus=12, gpus=0, mem=0.0/92.6 GB
	0.4285	 = Validation score   (r2)
	0.77s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: RandomForestMSE ...
	Fitting with cpus=24, gpus=0, mem=0.0/92.6 GB
	0.4197	 = Validation score   (r2)
	2.41s	 = Training   runtime
	0.15s	 = Validation runtime
Fitting model: CatBoost ...
	Fitting with cpus=12, gpus=0
	0.4397	 = Validation score   (r2)
	2.31s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: ExtraTreesMSE ...
	Fitting with cpus=24, gpus=0, mem=0.0/92.6 GB
	0.4155	 = Validation score   (r2)
	0.67s	 = Training   runtime
	0.13s	 = Validation runtime
Fitting model: NeuralNetFastAI ...
	Fitting with cpus=12, gpus=0, mem=0.0/92.5 GB
	0.3816	 = Validation score   (r2)
	4.91s	 = Training   runtime
	0.01s	 = Validation runtime
Fitting model: XGBoost 

In [78]:
autoglue_predictor_lrp.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val   fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.454802          r2       0.024068  18.708204                0.000574           0.060801            2       True         10
1             CatBoost   0.439702          r2       0.004760   2.309212                0.004760           2.309212            1       True          4
2           LightGBMXT   0.437989          r2       0.003294   1.103366                0.003294           1.103366            1       True          1
3       NeuralNetTorch   0.433011          r2       0.015440  15.234825                0.015440          15.234825            1       True          8
4             LightGBM   0.428459          r2       0.002545   0.774654                0.002545           0.774654            1       True          2
5      RandomForestMSE   0.419664     

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'NeuralNetFastAI': 'NNFastAiTabularModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.4379893392889389,
  'LightGBM': 0.4284585965185018,
  'RandomForestMSE': 0.41966368180492186,
  'CatBoost': 0.439702264460187,
  'ExtraTreesMSE': 0.4155029354075469,
  'NeuralNetFastAI': 0.3815885454406398,
  'XGBoost': 0.41270335853995377,
  'NeuralNetTorch': 0.4330105518269899,
  'LightGBMLarge': 0.4040548706159537,
  'WeightedEnsemble_L2': 0.45480179070655313},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  'Ne

In [ ]:
y_pred = autoglue_predictor_lrp.predict(X_test)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test - y_pred))
lrp_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Autogluon_Tab',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


R2 test score 0.4368
MAE test score 0.1516
MSE test score 0.0338


##### Images

In [80]:
X_train_img = pd.merge(X_train, img_path, left_index=True, right_index=True)
X_test_img = pd.merge(X_test, img_path, how="left", right_index=True, left_index=True)
train_lrp_img = pd.merge(X_train_img, y_train_lrp, left_index=True, right_index=True)

In [83]:
autoglue_model_lrp_img = MultiModalPredictor(label="lrp", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/lrp/img/")
hyperparams = {
    "env.strategy": "ddp_notebook",
    "env.num_gpus": 1,
    "env.num_workers": 0,
    #"optim.max_epochs": 15,
    #"optim.patience": 3,
    #"optim.learning_rate": 1e-4,
    #"optim.weight_decay": 1e-4,
    #"optim.warmup_steps": 500,
    #"env.per_gpu_batch_size": 8,
    #"env.batch_size": 128,
    #"model.timm_image.checkpoint_name": "convnext_base",
}
autoglue_predictor_lrp_img = autoglue_model_lrp_img.fit(train_lrp_img, hyperparameters=hyperparams)

=================== System Info ===================
AutoGluon Version:  1.4.1b20251210
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #88~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Oct 14 14:03:14 UTC 2
CPU Count:          24
Pytorch Version:    2.7.1+cu126
CUDA Version:       12.6
GPU Memory:         GPU 0: 23.69/23.69 GB | GPU 1: 23.69/23.69 GB
Total GPU Memory:   Free: 47.38 GB, Allocated: 0.00 GB, Total: 47.38 GB
GPU Count:          2
Memory Avail:       92.40 GB / 125.48 GB (73.6%)
Disk Space Avail:   9.64 GB / 3665.96 GB (0.3%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.

AutoMM starts to create your model. ✨✨✨

To track the learning progress, you can open a terminal and launch Tensorboard:
    ```shell
    # Assume you have installed tensorboard
    tensorboard --logdir /home/felix/Predictability-AD/models/assessors/autoglue_tab/lrp/img
    ```

Seed set to 0
GPU Count: 2
GPU Count 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Epoch 0:  50%|█████     | 29/58 [00:07<00:07,  4.07it/s]

...

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Predicting DataLoader 0: 100%|██████████| 4/4 [00:00<00:00,  7.88it/s]


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Predicting DataLoader 0: 100%|██████████| 4/4 [00:00<00:00,  7.53it/s]


AutoMM has created your model. 🎉🎉🎉

To load the model, use the code below:
    ```python
    from autogluon.multimodal import MultiModalPredictor
    predictor = MultiModalPredictor.load("/home/felix/Predictability-AD/models/assessors/autoglue_tab/lrp/img")
    ```

If you are not satisfied with the model, try to increase the training time, 
adjust the hyperparameters (https://auto.gluon.ai/stable/tutorials/multimodal/advanced_topics/customization.html),
or post issues on GitHub (https://github.com/autogluon/autogluon/issues).




In [ ]:
autoglue_predictor_lrp_img.fit_summary(verbosity=2)

In [ ]:
y_pred = autoglue_predictor_lrp_img.predict(X_test_img)
lrp_test_r2 = r2_score(y_test_lrp, y_pred)
lrp_test_mae = np.mean(np.abs(y_test - y_pred))
lrp_test_mse = np.mean((y_test - y_pred)**2)
print(f"R2 test score {lrp_test_r2:.4f}")
print(f"MAE test score {lrp_test_mae:.4f}")
print(f"MSE test score {lrp_test_mse:.4f}")
model_results.append({
    'target': 'LRP',
    'model': 'Autogluon_Mult',
    'test_r2': lrp_test_r2,
    'test_mae': lrp_test_mae,
    'test_mse': lrp_test_mse,
})


### Final Model Comparison


In [ ]:
results_df = pd.DataFrame(model_results, index=None)
results_df["test_r2"] = results_df["test_r2"].round(4)
results_df["test_mae"] = results_df["test_mae"].round(4)
results_df["test_mse"] = results_df["test_mse"].round(4)
results_df.to_csv("results.csv")
display(results_df)


,target,model,test_r2,test_mae,test_mse
0,IoU,Constant Mean Predictor,-0.0004,0.1227,0.0273
1,IoU,Univariate Linear Regression (Confidence),0.3493,0.0996,0.0177
2,IoU,Linear Regression,0.3679,0.0984,0.0172
3,IoU,Decision Tree,0.3425,0.0997,0.0179
4,IoU,Random Forest,0.3772,0.0969,0.0170
5,IoU,MLP,0.3279,0.1023,0.0183
6,IoU,XGBoost,0.3618,0.0997,0.0174
7,IoU,Autogluon_tab,0.3863,0.0970,0.0167
8,LRP,Constant Mean Predictor,-0.0002,0.1054,0.0206
9,LRP,Univariate Linear Regression (Confidence),0.3946,0.0822,0.0125


In [ ]:
print(results_df.to_latex(index=False))

\begin{tabular}{llrrr}
\toprule
target & model & test_r2 & test_mae & test_mse \\
\midrule
IoU & Constant Mean Predictor & -0.000400 & 0.122700 & 0.027300 \\
IoU & Univariate Linear Regression (Confidence) & 0.349300 & 0.099600 & 0.017700 \\
IoU & Linear Regression & 0.367900 & 0.098400 & 0.017200 \\
IoU & Decision Tree & 0.342500 & 0.099700 & 0.017900 \\
IoU & Random Forest & 0.377200 & 0.096900 & 0.017000 \\
IoU & MLP & 0.327900 & 0.102300 & 0.018300 \\
IoU & XGBoost & 0.361800 & 0.099700 & 0.017400 \\
IoU & Autogluon_tab & 0.386300 & 0.097000 & 0.016700 \\
LRP & Constant Mean Predictor & -0.000200 & 0.105400 & 0.020600 \\
LRP & Univariate Linear Regression (Confidence) & 0.394600 & 0.082200 & 0.012500 \\
LRP & Linear Regression & 0.420800 & 0.080200 & 0.011900 \\
LRP & Decision Tree & 0.410800 & 0.078500 & 0.012100 \\
LRP & Random Forest & 0.432900 & 0.077200 & 0.011700 \\
LRP & MLP & 0.405200 & 0.080500 & 0.012300 \\
LRP & XGBoost & 0.413000 & 0.079400 & 0.012100 \\
lrp & Autogluon